### Knowledge Graph builder (deontic, party-centric)

#### 1. Setup — locate repo, load API keys

In [1]:
import sys, os, json
from pathlib import Path
from collections import Counter

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'server').exists():
    ROOT = ROOT.parent
SERVER = ROOT / 'server'
if str(SERVER) not in sys.path:
    sys.path.insert(0, str(SERVER))

from dotenv import load_dotenv
load_dotenv(SERVER / '.env')

print('ROOT   :', ROOT)
print('OPENAI key set:', bool(os.getenv('OPENAI_API_KEY')))

ROOT   : /home/sante/Documents/FGV/me/SecondPaper
OPENAI key set: True


#### 2. Config — provider, input/output folders

In [2]:
PROVIDER = 'openai'   # only 'openai' (gpt-4.1) is wired; factory is extensible

PARAGRAPHS_DIR = ROOT / 'infra/json/paragraphs'
KG_OUT_DIR     = ROOT / 'infra/json/kg'
KG_OUT_DIR.mkdir(parents=True, exist_ok=True)

available = sorted(p.name for p in PARAGRAPHS_DIR.glob('*.json'))
for i, name in enumerate(available):
    print(i, name)

0 target_BELLICUMPHARMACEUTICALS_INC_05_07_2019-EX-10_1-Supply_Agreement.json
1 target_HealthcentralCom_19991108_S-1A_EX-10_27_6623292_EX-10_27_Co-Branding_Agreement.json
2 target_RitterPharmaceuticalsInc_20200313_S-4A_EX-10_54_12055220_EX-10_54_Development_Agreement.json
3 target_SteelVaultCorp_20081224_10-K_EX-10_16_3074935_EX-10_16_Affiliate_Agreement.json
4 target_TomOnlineInc_20060501_20-F_EX-4_46_749700_EX-4_46_Co-Branding_Agreement.json


#### 3. Load one document's paragraphs

In [3]:
DOC_INDEX = 0   # pick from the list above
DOC_FILE = available[DOC_INDEX]

src = json.load(open(PARAGRAPHS_DIR / DOC_FILE))
paragraphs = src['paragraphs']
doc_id = src['documentId']
print(doc_id)
print(len(paragraphs), 'paragraphs')

target::BELLICUMPHARMACEUTICALS_INC_05_07_2019-EX-10.1-Supply_Agreement
403 paragraphs


#### 4. Build the knowledge graph

Chunks the paragraphs, calls the LLM per chunk, and merges parties/clauses
across chunks (entity resolution). One LLM call per chunk — long contracts
take a bit.

In [4]:
from services.graph.knowledge.extraction import build_knowledge_graph
from services.llm.factory import LLMProviderFactory

provider = LLMProviderFactory.create(PROVIDER)
kg = build_knowledge_graph(paragraphs, provider)

NODE_KEYS = ['parties', 'clauses', 'definedTerms', 'provisions', 'conditions', 'references', 'values']
nodes = {key: len(getattr(kg, key)) for key in NODE_KEYS}
edges = Counter(e.type for e in kg.edges)

print(f'{"NODES":<24}{sum(nodes.values()):>6}')
for key, n in nodes.items():
    print(f'  {key:<22}{n:>6}')
    if key == 'provisions':
        for t, m in Counter(p.type for p in kg.provisions).most_common():
            print(f'    {t:<20}{m:>6}')

print(f'\n{"EDGES":<24}{len(kg.edges):>6}')
for t, n in edges.most_common():
    print(f'  {t:<22}{n:>6}')

# Coverage — a provision with no party never surfaces in any party's view.
no_party = [p for p in kg.provisions
            if not (p.beneficiaryPartyId if p.type == 'right' else p.obligorPartyId)]
no_clause = [p for p in kg.provisions if not p.clauseId]
total = max(1, len(kg.provisions))
semantic = edges['references'] + edges['depends_on'] + edges['supersedes'] + edges['modifies']

print(f'\nprovisions with no party : {len(no_party):>4}  ({100 * len(no_party) / total:.0f}%)')
print(f'provisions with no clause: {len(no_clause):>4}  ({100 * len(no_clause) / total:.0f}%)')
print(f'clause<->clause edges    : {semantic:>4}  (references + depends_on + supersedes + modifies)')

NODES                      783
  parties                    5
  clauses                  142
  definedTerms             111
  provisions               372
    obligation             250
    right                   63
    prohibition             59
  conditions                71
  references                31
  values                    51

EDGES                     1083
  is_part_of               556
  assigns_obligation_to    240
  uses                     111
  defines                   68
  references                50
  grants_right_to           44
  depends_on                14

provisions with no party :   88  (24%)
provisions with no clause:   50  (13%)
clause<->clause edges    :   64  (references + depends_on + supersedes + modifies)


#### 5. Save the KG

In [5]:
out_path = KG_OUT_DIR / DOC_FILE
out_path.write_text(json.dumps(kg.model_dump(), ensure_ascii=False, indent=2), encoding='utf-8')
print('saved:', out_path)

saved: /home/sante/Documents/FGV/me/SecondPaper/infra/json/kg/target_BELLICUMPHARMACEUTICALS_INC_05_07_2019-EX-10_1-Supply_Agreement.json


#### 6. Inspect — parties and sample provisions

In [6]:
for p in kg.parties:
    print(f'[{p.id}] {p.name!r}  role={p.role!r}  aliases={p.aliases}')
print()
for pv in kg.provisions[:10]:
    print(f'[{pv.id}] {pv.type:11s} obligor={pv.obligorPartyId} benef={pv.beneficiaryPartyId} clause={pv.clauseId}')
    print('    ', pv.summary)

[party-1] 'Miltenyi Biotec GmbH'  role='Supplier'  aliases=['Miltenyi', 'Miltenyi Affiliate', 'Miltenyi Indemnitee', 'Party']
[party-2] 'Bellicum Pharmaceuticals, Inc.'  role='Buyer'  aliases=['Bellicum', 'Bellicum Indemnitee', 'Party']
[party-3] 'Second-Source Supplier'  role='Alternate Manufacturer'  aliases=[]
[party-4] 'Receiving Party'  role='Receiving Party'  aliases=[]
[party-5] 'Disclosing Party'  role='Disclosing Party'  aliases=[]

[perm-1] right       obligor=None benef=party-2 clause=None
     Bellicum may use certain Miltenyi Products solely for the Permitted Use in connection with development and manufacture of Bellicum Products.
[obl-1] obligation  obligor=party-1 benef=party-2 clause=None
     Miltenyi must sell Miltenyi Products to Bellicum in accordance with the Agreement.
[obl-2] obligation  obligor=party-2 benef=party-1 clause=None
     Bellicum must purchase Miltenyi Products from Miltenyi in accordance with the Agreement.
[obl-3] obligation  obligor=party-1 benef=

#### 7. (Optional) Batch — one KG per contract, for every file

Processes all documents in `infra/json/paragraphs/` at once. Each iteration
builds **one** knowledge graph for a whole contract (the internal chunking is
merged), and writes one KG file per contract to `infra/json/kg/`.

In [7]:
# for f in available:
#     s = json.load(open(PARAGRAPHS_DIR / f))
#     g = build_knowledge_graph(s['paragraphs'], provider)
#     (KG_OUT_DIR / f).write_text(json.dumps(g.model_dump(), ensure_ascii=False, indent=2), encoding='utf-8')
#     print(f'{f[:55]:57s} provisions={len(g.provisions):4d} parties={len(g.parties)}')